# Tarea 8 · pruebas aisladas del MERGE

Crea dos tablas de prueba Delta con nombre único en el esquema Silver. No altera las tablas operativas, Bronze ni sus checkpoints. Las tablas de prueba se conservan para inspección; este notebook no las elimina. Antes de repetir las pruebas, ejecutar de nuevo la celda de preparación para generar nombres nuevos.


In [1]:
import json
import sys
from datetime import date, datetime, timezone
from pathlib import Path
from pyspark.sql import functions as F

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'config' / 'silver.json').is_file()), None)
if project_root is None:
    raise FileNotFoundError('No se encuentra config/silver.json')
sys.path.insert(0, str(project_root / 'scripts'))
from merge_mediciones import merge_candidatas, historial_merge
silver = json.loads((project_root / 'config' / 'silver.json').read_text(encoding='utf-8'))
suffix = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
prefix = '.'.join(silver['tables']['mediciones_vigentes'].split('.')[:2])
test_target = f'{prefix}.mediciones_merge_prueba_{suffix}'
test_conflicts = f'{prefix}.conflictos_merge_prueba_{suffix}'
schema = spark.table(silver['tables']['mediciones']).schema
print('Tablas de prueba:', test_target, test_conflicts)


Tablas de prueba: workspace.lensdata_silver.mediciones_merge_prueba_20261126_151948_376537 workspace.lensdata_silver.conflictos_merge_prueba_20261126_151948_376537


In [2]:
def fila(version, reproducciones, archivo):
    return {'plataforma': 'tiktok', 'video_id': 'V000',
            'fecha_medicion': date(2026, 11, 1), 'version_origen': version,
            'reproducciones': reproducciones, 'me_gusta': 10,
            'comentarios': 2, 'compartidos': 1,
            '_source_file': f'prueba/tarea08/{archivo}.csv',
            '_processed_at': datetime.now(timezone.utc), '_quality_warnings': []}

def aplicar(nombre, *filas, inserted=0, updated=0):
    before = historial_merge(spark, test_target, limit=1).collect()[0].version if spark.catalog.tableExists(test_target) else None
    entrada = spark.createDataFrame(list(filas), schema=schema)
    resultado = merge_candidatas(spark, entrada, test_target, test_conflicts, silver['incremental_merge'])
    vigente = spark.table(test_target).collect()
    assert len(vigente) == 1
    after = historial_merge(spark, test_target, limit=1).collect()[0]
    if after.version != before:
        assert after.operation == 'MERGE'
        metrics = after.operationMetrics or {}
        if 'numTargetRowsInserted' in metrics:
            assert int(metrics['numTargetRowsInserted']) == inserted
        if 'numTargetRowsUpdated' in metrics:
            assert int(metrics['numTargetRowsUpdated']) == updated
    else:
        assert inserted == updated == 0
    print(nombre, resultado, 'vigente:', vigente[0].version_origen, vigente[0].reproducciones, vigente[0]._source_file, 'métricas:', after.operationMetrics)
    return vigente[0]

original = fila(1, 100, 'original')
assert aplicar('inserción', original, inserted=1).version_origen == 1
assert aplicar('reenvío idéntico', fila(1, 100, 'reenvio'))._source_file == original['_source_file']
correccion = fila(2, 90, 'correccion')
assert aplicar('actualización', correccion, updated=1).reproducciones == 90
assert aplicar('versión antigua tardía', fila(1, 100, 'antigua_tardia')).version_origen == 2
assert aplicar('conflicto con destino', fila(2, 91, 'conflicto')).reproducciones == 90
assert aplicar('idempotencia', correccion)._source_file == correccion['_source_file']
assert aplicar('conflicto dentro del lote', fila(3, 80, 'v3_a'), fila(3, 81, 'v3_b')).version_origen == 2
assert spark.table(test_conflicts).filter(F.col('conflict_code') == 'SAME_VERSION_DIFFERS_FROM_TARGET').count() == 1
assert spark.table(test_conflicts).filter(F.col('conflict_code') == 'SAME_KEY_VERSION_DIFFERENT_METRICS').count() == 2
display(historial_merge(spark, test_target, limit=20))


inserción {'candidatas': 1, 'reenvios_en_lote': 0, 'conflictos_en_lote': 0, 'versiones_superadas_en_lote': 0, 'conflictos_con_destino': 0} vigente: 1 100 prueba/tarea08/original.csv métricas: {'numTargetRowsCopied': '0', 'numTargetRowsDeleted': '0', 'numTargetBytesRemoved': '0', 'numTargetDeletionVectorsAdded': '0', 'numTargetRowsMatchedUpdated': '0', 'numTargetRowsMatchedDeleted': '0', 'numTargetRowsUpdated': '0', 'numTargetChangeFilesAdded': '0', 'numTargetRowsNotMatchedBySourceDeleted': '0', 'rewriteTimeMs': '982', 'numTargetFilesAdded': '1', 'numTargetBytesAdded': '3786', 'executionTimeMs': '2326', 'materializeSourceTimeMs': '657', 'numTargetRowsInserted': '1', 'numTargetDeletionVectorsUpdated': '0', 'scanTimeMs': '632', 'numOutputRows': '1', 'numTargetDeletionVectorsRemoved': '0', 'numTargetRowsNotMatchedBySourceUpdated': '0', 'numSourceRows': '1', 'numTargetFilesRemoved': '0'}
reenvío idéntico {'candidatas': 1, 'reenvios_en_lote': 0, 'conflictos_en_lote': 0, 'versiones_superadas_

,version,timestamp,operation,operationMetrics
0,7,2026-11-26 15:22:06.743,MERGE,"{'numTargetRowsCopied': '0', 'numTargetRowsDeleted': '0', 'numTargetBytesRemoved': '0', 'numTargetDeletionVectorsAdded': '0', 'numTargetRowsMatchedUpdated': '0', 'numTargetRowsMatchedDeleted': '0', 'numTargetRowsUpdated': '0', 'numTargetChangeFilesAdded': '0', 'numTargetRowsNotMatchedBySourceDeleted': '0', 'rewriteTimeMs': '828', 'numTargetFilesAdded': '0', 'numTargetBytesAdded': '0', 'executionTimeMs': '2015', 'materializeSourceTimeMs': '529', 'numTargetRowsInserted': '0', 'numTargetDeletionVectorsUpdated': '0', 'scanTimeMs': '619', 'numOutputRows': '0', 'numTargetDeletionVectorsRemoved': '0', 'numTargetRowsNotMatchedBySourceUpdated': '0', 'numSourceRows': '0', 'numTargetFilesRemoved': '0'}"
1,6,2026-11-26 15:21:47.479,MERGE,"{'numTargetRowsCopied': '0', 'numTargetRowsDeleted': '0', 'numTargetBytesRemoved': '0', 'numTargetDeletionVectorsAdded': '0', 'numTargetRowsMatchedUpdated': '0', 'numTargetRowsMatchedDeleted': '0', 'numTargetRowsUpdated': '0', 'numTargetChangeFilesAdded': '0', 'numTargetRowsNotMatchedBySourceDeleted': '0', 'rewriteTimeMs': '1251', 'numTargetFilesAdded': '0', 'numTargetBytesAdded': '0', 'executionTimeMs': '2608', 'materializeSourceTimeMs': '526', 'numTargetRowsInserted': '0', 'numTargetDeletionVectorsUpdated': '0', 'scanTimeMs': '792', 'numOutputRows': '0', 'numTargetDeletionVectorsRemoved': '0', 'numTargetRowsNotMatchedBySourceUpdated': '0', 'numSourceRows': '1', 'numTargetFilesRemoved': '0'}"
2,5,2026-11-26 15:21:30.042,MERGE,"{'numTargetRowsCopied': '0', 'numTargetRowsDeleted': '0', 'numTargetBytesRemoved': '0', 'numTargetDeletionVectorsAdded': '0', 'numTargetRowsMatchedUpdated': '0', 'numTargetRowsMatchedDeleted': '0', 'numTargetRowsUpdated': '0', 'numTargetChangeFilesAdded': '0', 'numTargetRowsNotMatchedBySourceDeleted': '0', 'rewriteTimeMs': '1206', 'numTargetFilesAdded': '0', 'numTargetBytesAdded': '0', 'executionTimeMs': '2612', 'materializeSourceTimeMs': '536', 'numTargetRowsInserted': '0', 'numTargetDeletionVectorsUpdated': '0', 'scanTimeMs': '840', 'numOutputRows': '0', 'numTargetDeletionVectorsRemoved': '0', 'numTargetRowsNotMatchedBySourceUpdated': '0', 'numSourceRows': '1', 'numTargetFilesRemoved': '0'}"
3,4,2026-11-26 15:21:07.516,MERGE,"{'numTargetRowsCopied': '0', 'numTargetRowsDeleted': '0', 'numTargetBytesRemoved': '0', 'numTargetDeletionVectorsAdded': '0', 'numTargetRowsMatchedUpdated': '0', 'numTargetRowsMatchedDeleted': '0', 'numTargetRowsUpdated': '0', 'numTargetChangeFilesAdded': '0', 'numTargetRowsNotMatchedBySourceDeleted': '0', 'rewriteTimeMs': '1242', 'numTargetFilesAdded': '0', 'numTargetBytesAdded': '0', 'executionTimeMs': '2611', 'materializeSourceTimeMs': '512', 'numTargetRowsInserted': '0', 'numTargetDeletionVectorsUpdated': '0', 'scanTimeMs': '824', 'numOutputRows': '0', 'numTargetDeletionVectorsRemoved': '0', 'numTargetRowsNotMatchedBySourceUpdated': '0', 'numSourceRows': '1', 'numTargetFilesRemoved': '0'}"
4,3,2026-11-26 15:20:48.960,MERGE,"{'numTargetRowsCopied': '0', 'numTargetRowsDeleted': '0', 'numTargetBytesRemoved': '3786', 'numTargetDeletionVectorsAdded': '0', 'numTargetRowsMatchedUpdated': '1', 'numTargetRowsMatchedDeleted': '0', 'numTargetRowsUpdated': '1', 'numTargetChangeFilesAdded': '0', 'numTargetRowsNotMatchedBySourceDeleted': '0', 'rewriteTimeMs': '1383', 'numTargetFilesAdded': '1', 'numTargetBytesAdded': '4557', 'executionTimeMs': '2725', 'materializeSourceTimeMs': '484', 'numTargetRowsInserted': '0', 'numTargetDeletionVectorsUpdated': '0', 'scanTimeMs': '819', 'numOutputRows': '1', 'numTargetDeletionVectorsRemoved': '0', 'numTargetRowsNotMatchedBySourceUpdated': '0', 'numSourceRows': '1', 'numTargetFilesRemoved': '1'}"
5,2,2026-11-26 15:20:30.563,MERGE,"{'numTargetRowsCopied': '0', 'numTargetRowsDeleted': '0', 'numTargetBytesRemoved': '0', 'numTargetDeletionVectorsAdded': '0', 'numTargetRowsMatchedUpdated': '0', 'numTargetRowsMatchedDeleted': '0', 'numTargetRowsUpdated': '0